# 05 - Anomaly Detection

**CASEFILE: AI-Powered Missing Person Investigation System**  
*Phase 5: Trajectory Anomaly Detection & Kinematic Outlier Analysis*

---

### Overview
During a missing person investigation, analyzing GPS trajectory data for anomalous movements provides critical insights into unexpected route deviations, rapid speed shifts, or temporal departures from habitual mobility patterns. 

This notebook demonstrates:
1. **Tri-Model Anomaly Architecture**: Evaluating three unsupervised anomaly detection estimators:
   - **Isolation Forest (IF)**: Tree-based recursive space partitioning.
   - **Local Outlier Factor (LOF)**: Density-based nearest-neighbor reachability.
   - **One-Class Support Vector Machine (OC-SVM)**: Non-linear kernel boundary estimation.
2. **Ensemble Voting Mechanism**: Combining detector predictions using a **2-of-3 consensus vote** to minimize false positives.
3. **Kinematic Feature Profiling**: Analyzing velocity, step distance, and angular bearing divergence between normal and anomalous segments.
4. **Geospatial Mapping**: Visualizing anomaly spatial distributions across Beijing tracking coordinates.

> **ETHICAL DISCLAIMER: STATISTICAL ANOMALY != SUSPICIOUS OR CRIMINAL BEHAVIOR**  
> An anomalous GPS waypoint signifies mathematical divergence from historical mobility baselines (e.g., sudden accelerations, unfamiliar transit corridors, atypical travel hours). **It does NOT imply criminal involvement, guilt, or suspicious activity.** In real-world missing person scenarios, anomalies frequently arise from benign events: public transit rerouting, detours due to urban construction, GPS signal multipath interference, or running an unexpected personal errand. Predictive scores serve strictly as heuristic search radius indicators.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Setup visualization styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# File paths
DATA_DIR = '../data' if os.path.exists('../data') else 'data'
REPORTS_DIR = '../reports' if os.path.exists('../reports') else 'reports'
MODELS_DIR = '../models' if os.path.exists('../models') else 'models'

print("Anomaly detection environment initialized successfully.")
print(f"Data Directory: {DATA_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")

## 1. Load GPS Telemetry (Efficient Subsampling)

The full processed anomaly dataset (`gps_anomalies.csv`) contains approximately **896,800 GPS tracking points** (~257 MB). To ensure fast interactive rendering and optimal memory usage, we load a representative sample of **10,000 records** (`nrows=10000`).

In [ ]:
import sys
sys.path.insert(0, '..')

anomalies_path = os.path.join(DATA_DIR, 'processed', 'gps_anomalies.csv')

# Load 10,000 rows for interactive exploration
df_sample = pd.read_csv(anomalies_path, nrows=10000)

print(f"Loaded subsample shape: {df_sample.shape}")
print(f"Feature columns: {df_sample.columns.tolist()[:10]} ...")

display(df_sample[['timestamp', 'speed_kmh', 'distance_km', 'bearing_change', 
                   'hour', 'if_anomaly', 'lof_anomaly', 'svm_anomaly', 
                   'ensemble_anomaly', 'anomaly_score']].head(5))

## 2. Multi-Model Anomaly Detection Results

Three unsupervised algorithms were fitted with a targeted contamination threshold of $\sim 5\%$:
1. **Isolation Forest**: Partitions feature space (`speed_kmh`, `distance_km`, `hour`, `bearing_change`, `time_delta_seconds`) using randomized trees.
2. **Local Outlier Factor (LOF)**: Identifies points whose local density is significantly lower than that of their $k=20$ nearest neighbors.
3. **One-Class SVM**: Fits a support vector boundary around the high-density training manifold using an RBF kernel.

Let's compare the empirical anomaly detection rates across all three algorithms.

In [ ]:
import sys
sys.path.insert(0, '..')

total_pts = len(df_sample)
if_count = int(df_sample['if_anomaly'].sum())
lof_count = int(df_sample['lof_anomaly'].sum())
svm_count = int(df_sample['svm_anomaly'].sum())
ens_count = int(df_sample['ensemble_anomaly'].sum())

model_rates = pd.DataFrame({
    'Detection Algorithm': [
        'Isolation Forest',
        'Local Outlier Factor (LOF)',
        'One-Class SVM',
        'Ensemble Consensus (>= 2 votes)'
    ],
    'Anomalies Flagged': [if_count, lof_count, svm_count, ens_count],
    'Anomaly Rate (%)': [
        f"{(if_count / total_pts) * 100:.2f}%",
        f"{(lof_count / total_pts) * 100:.2f}%",
        f"{(svm_count / total_pts) * 100:.2f}%",
        f"{(ens_count / total_pts) * 100:.2f}%"
    ],
    'Target Contamination': ['5.0%', '5.0%', '5.0%', 'Consensus Filter'],
    'Primary Mechanism': [
        'Recursive tree splits (short path length)',
        'Local reachability density drop vs neighbors',
        'Non-linear RBF kernel boundary distance',
        'Majority agreement filter (reduces false alarms)'
    ]
})

print("=" * 65)
print("       ANOMALY DETECTOR DETECTION RATE COMPARISON")
print("=" * 65)
display(model_rates)

## 3. Ensemble Voting Mechanism (2-of-3 Agreement)

Single-model anomaly detectors in urban GPS streams are susceptible to false alarms triggered by tall building multipath, tunnel exit jumps, or temporary traffic light stops.

To achieve high precision, CASEFILE employs a **2-of-3 consensus ensemble**:
$$\text{Ensemble Anomaly} = \mathbb{I}\left(\text{IF} + \text{LOF} + \text{SVM} \ge 2\right)$$

The continuous `anomaly_score` is the normalized average of the inverted decision function scores across all three models:
$$\text{Score} = \frac{S_{\text{IF}} + S_{\text{LOF}} + S_{\text{SVM}}}{3} \in [0.0, 1.0]$$

In [ ]:
import sys
sys.path.insert(0, '..')

# Compute voting distribution across the sample
vote_sum = df_sample['if_anomaly'] + df_sample['lof_anomaly'] + df_sample['svm_anomaly']
vote_distribution = vote_sum.value_counts().sort_index()

agreement_df = pd.DataFrame({
    'Detector Agreement Level': [
        '0 Detectors (Consensus Normal)',
        '1 Detector (Discarded Single-Model False Positive)',
        '2 Detectors (Ensemble Anomaly - High Likelihood)',
        '3 Detectors (Ensemble Anomaly - Highest Confidence)'
    ],
    'Point Count': [vote_distribution.get(i, 0) for i in range(4)],
    'Proportion (%)': [f"{(vote_distribution.get(i, 0) / total_pts) * 100:.2f}%" for i in range(4)],
    'Ensemble Action': [
        'Classified as Normal (0)',
        'Filtered out as Noise (0)',
        'Flagged as Anomaly (1)',
        'Flagged as Anomaly (1)'
    ]
})

print("=" * 65)
print("          ENSEMBLE VOTING CONSENSUS BREAKDOWN")
print("=" * 65)
display(agreement_df)

## 4. Kinematic Profile Comparison: Normal vs. Anomalous Trajectories

We compare the kinematic feature distributions between routine trajectory segments (`ensemble_anomaly == 0`) and detected anomalies (`ensemble_anomaly == 1`).

In [ ]:
import sys
sys.path.insert(0, '..')

normal_df = df_sample[df_sample['ensemble_anomaly'] == 0]
anom_df = df_sample[df_sample['ensemble_anomaly'] == 1]

features = ['speed_kmh', 'distance_km', 'bearing_change']
stats_rows = []

for f in features:
    stats_rows.append({
        'Kinematic Feature': f,
        'Normal Mean': f"{normal_df[f].mean():.2f}",
        'Normal Median': f"{normal_df[f].median():.2f}",
        'Normal Std': f"{normal_df[f].std():.2f}",
        'Anomalous Mean': f"{anom_df[f].mean():.2f}",
        'Anomalous Median': f"{anom_df[f].median():.2f}",
        'Anomalous Std': f"{anom_df[f].std():.2f}"
    })

print("=" * 75)
print("         KINEMATIC COMPARISON: NORMAL vs ANOMALOUS MOVEMENTS")
print("=" * 75)
display(pd.DataFrame(stats_rows))

In [ ]:
import sys
sys.path.insert(0, '..')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Speed Comparison Boxplot
sns.boxplot(x='ensemble_anomaly', y='speed_kmh', data=df_sample, ax=ax1, 
            palette=['#3498db', '#e74c3c'], showfliers=False)
ax1.set_xticklabels(['Normal Points (0)', 'Ensemble Anomalies (1)'])
ax1.set_title("Speed Comparison: Normal vs Anomalous", fontweight='bold')
ax1.set_xlabel("Classification")
ax1.set_ylabel("Speed (km/h)")
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 2. Continuous Anomaly Score Distribution
ax2.hist(normal_df['anomaly_score'], bins=35, alpha=0.6, color='#3498db', 
         density=True, label='Normal Points')
ax2.hist(anom_df['anomaly_score'], bins=35, alpha=0.75, color='#e74c3c', 
         density=True, label='Ensemble Anomalies')
ax2.set_title("Ensemble Anomaly Score Distribution", fontweight='bold')
ax2.set_xlabel("Aggregated Anomaly Score (0.0 to 1.0)")
ax2.set_ylabel("Density")
ax2.legend(loc='upper right')
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 5. Pipeline Anomaly Distribution Report

We inspect the multi-feature anomaly distribution plot generated during the full pipeline execution on the complete dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

dist_report = os.path.join(REPORTS_DIR, 'anomaly_distributions.png')
if os.path.exists(dist_report):
    print("Loading reports/anomaly_distributions.png:")
    display(Image(filename=dist_report, width=820))
else:
    print(f"Warning: {dist_report} not found.")

## 6. Spatial Mapping of Anomalous GPS Waypoints

We plot the geographic distribution of normal points versus consensus anomalous points across the metropolitan coordinates.

In [ ]:
import sys
sys.path.insert(0, '..')

plt.figure(figsize=(11, 8))

# Plot normal points in translucent blue
plt.scatter(normal_df['longitude'], normal_df['latitude'], 
            c='#3498db', s=14, alpha=0.4, label=f"Normal Points (n={len(normal_df):,})")

# Plot ensemble anomalous points in highlighted red
plt.scatter(anom_df['longitude'], anom_df['latitude'], 
            c='#e74c3c', marker='x', s=45, linewidths=1.5, alpha=0.95, 
            label=f"Ensemble Anomalies (n={len(anom_df):,})")

plt.title("Spatial Distribution of Trajectory Anomalies in Beijing Coordinates", fontsize=13, fontweight='bold')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(loc='upper right', frameon=True, framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Operational Utility & Investigative Synthesis

- **Point of Disappearance (POD) Identification**: When a missing person's trajectory abruptly transitions from normal scores ($<0.2$) to anomalous scores ($>0.7$) before signal termination, that inflection point marks the primary tactical search epicenter.
- **False Positive Filtering**: Solo detectors generated a 5% outlier rate independently, but the 2-of-3 ensemble consensus filtered out single-sensor glitches, preserving high-fidelity behavioral shifts.
- **Search Radius Adaptation**: Segments characterized by high anomalous velocity trigger an expanded search radius model, while stationary anomalous dwell points trigger localized high-density sweeps.